<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/scikit-learn/Bank_Load_Default_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ushbu sahifada mijoz bank berayotgan kreditni qaytara oladimi yoki yo'qmi bashorat qiluvchi model yaratamiz . Dataset AI tomonidan real hayotdagidek qilib yaratiladi.**

- target ustun `default` : 0-kreditni o'z vaqtida to'laydi , 1-kreditni qaytara olmaydi

In [1]:
import numpy as np
import pandas as pd
import sklearn

In [2]:
df = pd.DataFrame({
    "age": [25, 45, 35, 29, 50, 41, 38, 27, 33, 48,
            30, 36, 52, 28, 40, 31, 46, 34, 43, 26],

    "income": [3000, 9000, 6000, np.nan, 12000, 8000, 7000, 3500, 5000, 11000,
               4500, 6500, 13000, 4000, 8500, 4800, 10000, 5500, 9200, 3200],

    "loan_amount": [5000, 15000, 8000, 6000, 20000, 12000, 10000, 5500, 7500, 18000,
                    7000, 9500, 22000, 6500, 11000, 7200, 17000, 8500, 14000, 5200],

    "job_type": [
        "Private","Government","Private","Private","Business",
        "Government","Private","Private","Business","Government",
        "Private","Business","Government","Private","Business",
        "Private","Government","Business","Government","Private"
    ],

    "married": [
        "Yes","Yes","No","No","Yes",
        "Yes","No","No","Yes","Yes",
        "No","Yes","Yes","No","Yes",
        "No","Yes","No","Yes","No"
    ],

    "default": [
        1,0,0,1,0,
        0,0,1,0,0,
        1,0,0,1,0,
        1,0,0,0,1
    ]
})
df.head()

,age,income,loan_amount,job_type,married,default
0,25,3000.0,5000,Private,Yes,1
1,45,9000.0,15000,Government,Yes,0
2,35,6000.0,8000,Private,No,0
3,29,NaN,6000,Private,No,1
4,50,12000.0,20000,Business,Yes,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   age          20 non-null     int64  
 1   income       19 non-null     float64
 2   loan_amount  20 non-null     int64  
 3   job_type     20 non-null     object 
 4   married      20 non-null     object 
 5   default      20 non-null     int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 1.1+ KB


In [4]:
# X va y larni ajratish
X = df.drop('default', axis=1)
y = df['default']

In [5]:
# sonli va matnli ustunlarni olish
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=np.object_).columns.tolist()

In [6]:
num_cols

['age', 'income', 'loan_amount']

In [7]:
cat_cols

['job_type', 'married']

In [8]:
from sklearn import preprocessing
# kerakli paketlarni chaqirish
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# sonli ustunlar uchun pipeline yaratish
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# kategoriyali ustunlar uchun pipeline
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# sonli va matnli pipelinelarni birlashtirish
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# to'liq pipeline yaratish + model
full_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression())
])

In [18]:
# cross-validation qilish
from sklearn.model_selection import cross_validate, StratifiedKFold


# balansni saqlagan holda cross-validation qilish
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

score = cross_validate(full_pipeline, X, y, cv=cv, scoring=["accuracy", "precision", "recall", "f1"])

print(score)

{'fit_time': array([0.01580811, 0.01873732, 0.01653767, 0.01627517, 0.01652908]), 'score_time': array([0.02337813, 0.02097297, 0.02077579, 0.0203681 , 0.02085137]), 'test_accuracy': array([0.75, 1.  , 1.  , 1.  , 1.  ]), 'test_precision': array([0.66666667, 1.        , 1.        , 1.        , 1.        ]), 'test_recall': array([1., 1., 1., 1., 1.]), 'test_f1': array([0.8, 1. , 1. , 1. , 1. ])}


In [19]:
score["test_accuracy"].mean()

np.float64(0.95)

In [20]:
score["test_precision"].mean()

np.float64(0.9333333333333332)

In [21]:
score["test_recall"].mean()

np.float64(1.0)

In [22]:
score["test_f1"].mean()

np.float64(0.96)

In [23]:
score["test_accuracy"].std()

np.float64(0.09999999999999999)

In [24]:
score["test_precision"].std()

np.float64(0.13333333333333336)

In [25]:
score["test_recall"].std()

np.float64(0.0)

In [26]:
score["test_f1"].std()

np.float64(0.07999999999999999)

###**Xulosa**

- Model umumiy holatda juda yaxshi ishlatayapti . Precision - model kreditni qaytara olmaydi deganlarning 93% ni to'g'ri aniqlagan bu juda yaxshi . Kreditni qaytara olmaydiganlarni o'tkazib yubormayapti , bu ham juda yaxshi natija (recall) . F1 precision va recall muvozanati juda yaxshi